In [80]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [81]:
model = ChatOpenAI(model='gpt-3.5-turbo')

In [82]:
parser = JsonOutputParser()

In [83]:
prompt = PromptTemplate(template='Provide me with the Name, Age, City of a fictional person in {specific_format}', input_variable=[], partial_variables = {"specific_format" :parser.get_format_instructions()})

In [84]:
prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={'specific_format': 'Return a JSON object.'}, template='Provide me with the Name, Age, City of a fictional person in {specific_format}')

In [85]:
final_prompt = prompt.format()

In [86]:
final_prompt

'Provide me with the Name, Age, City of a fictional person in Return a JSON object.'

In [87]:
response = model.invoke(final_prompt)

In [88]:
response

AIMessage(content='{\n  "name": "Emily Smith",\n  "age": 28,\n  "city": "Springfield"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 26, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDOEaAQKQKXV4uBlTsRKfcpW61Sas', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a00929-ae09-7800-82a9-68527df9d1bf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 25, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [89]:
response.content

'{\n  "name": "Emily Smith",\n  "age": 28,\n  "city": "Springfield"\n}'

In [90]:
chain = prompt | model | parser

In [91]:
result = chain.invoke({})

In [92]:
result

{'Name': 'Sarah Johnson', 'Age': 32, 'City': 'Seattle'}

In [93]:
r = parser.parse(response.content)

In [94]:
r

{'name': 'Emily Smith', 'age': 28, 'city': 'Springfield'}

In [97]:
r['name']

'Emily Smith'

## Give me 5 Facts about this Topic

- `Pydantic Output Parser`: enforce a schema as the Required Reponse out of your LLM
- It ensures LLM Responses forms a well defined structure
- Automatically convert LLM ouputs - python object
- uses pydantic built in validation to catch incorrect or missing data

In [106]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [107]:
class Data(BaseModel):
    name: str = Field(description='return the name of the user')
    age: int = Field( gt=21, description = 'return the age of the user')
    city: str = Field(description='the city has to be of Bihar State'),
    state: str= Field(description='the sate has to be bihar only')

In [108]:
parser = PydanticOutputParser(pydantic_object = Data)

In [109]:
parser

PydanticOutputParser(pydantic_object=<class '__main__.Data'>)

In [110]:
prompt = PromptTemplate(
    template='Provide the Name, Age, City and State of the Fictional Person from this {nationality} in {format}',
    input_variable=['nationality'], 
    partial_variables={'format': parser.get_format_instructions})

In [111]:
prompt

PromptTemplate(input_variables=['nationality'], input_types={}, partial_variables={'format': <bound method PydanticOutputParser.get_format_instructions of PydanticOutputParser(pydantic_object=<class '__main__.Data'>)>}, template='Provide the Name, Age, City and State of the Fictional Person from this {nationality} in {format}')

In [114]:
final_prompt = prompt.invoke({'nationality':'Indian'})

In [116]:
final_prompt

StringPromptValue(text='Provide the Name, Age, City and State of the Fictional Person from this Indian in The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "return the name of the user", "title": "Name", "type": "string"}, "age": {"description": "return the age of the user", "exclusiveMinimum": 21, "title": "Age", "type": "integer"}, "city": {"title": "City", "type": "string"}, "state": {"description": "the sate has to be bihar only", "title": "State", "type": "string"}}, "required": ["name", "age", "state"]}\n```')

In [117]:
response = model.invoke(final_prompt)

In [118]:
response

AIMessage(content='{\n    "name": "Rajesh Kumar",\n    "age": 30,\n    "city": "Patna",\n    "state": "Bihar"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 262, 'total_tokens': 297, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EDOFxSIuQAMpSQ93mifqJjJUPDW3J', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0092a-f85c-7e51-b8d1-ea92887555b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 262, 'output_tokens': 35, 'total_tokens': 297, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [119]:
response.content

'{\n    "name": "Rajesh Kumar",\n    "age": 30,\n    "city": "Patna",\n    "state": "Bihar"\n}'

In [120]:
parser.parse(response.content)

Data(name='Rajesh Kumar', age=30, city='Patna', state='Bihar')

In [121]:
chain = prompt | model | parser

In [122]:
result = chain.invoke({'nationality':'Indian'})

c:\Users\arunk\anaconda3\envs\venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='the city has to be of Bihar State'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [123]:
result

Data(name='Rahul Kumar', age=25, city='Patna', state='Bihar')